In [0]:
# Test Python Code
message = "Hello Shoaib, Databricks is ready!"
print(message)



Hello Shoaib, Databricks is ready!


In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp

# 1. API Configuration
API_KEY = "21c5303e92e544fdf03028622b9ab242"  # <-- Replace with your key
CITY = "Lahore"
URL = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

# 2. Ingest Data from OpenWeather API
response = requests.get(URL)

if response.status_code == 200:
    data = response.json()
    
    # Extract & Structure Key Weather Metrics
    weather_data = [{
        "city": data["name"],
        "country": data["sys"]["country"],
        "temp_celsius": data["main"]["temp"],
        "feels_like_celsius": data["main"]["feels_like"],
        "humidity_pct": data["main"]["humidity"],
        "weather_condition": data["weather"][0]["main"],
        "wind_speed_m_s": data["wind"]["speed"]
    }]
    
    # 3. Create PySpark DataFrame & Add Ingestion Timestamp
    df = spark.createDataFrame(weather_data)
    df_raw = df.withColumn("ingested_at", current_timestamp())
    
    # Display the DataFrame
    display(df_raw)
else:
    print(f"API Request Failed. HTTP Status Code: {response.status_code}")

city,country,feels_like_celsius,humidity_pct,temp_celsius,weather_condition,wind_speed_m_s,ingested_at
Lahore,PK,40.27,49,34.99,Clear,2.06,2026-09-08T12:50:45.056Z


In [0]:
# Save DataFrame as a persistent Delta Lake table
delta_table_path = "dbfs:/user/hive/warehouse/lahore_weather_bronze"

df_raw.write.format("delta") \
    .mode("append") \
    .save(delta_table_path)

print("Successfully written Lahore weather data to Delta Lake Bronze table!")

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

# 1. API Request
API_KEY = "21c5303e92e544fdf03028622b9ab242"  # <-- Paste your actual OpenWeather API key
CITY = "Lahore"
URL = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

response = requests.get(URL)

if response.status_code == 200:
    data = response.json()
    
    weather_data = [{
        "city": data["name"],
        "country": data["sys"]["country"],
        "temp_celsius": data["main"]["temp"],
        "feels_like_celsius": data["main"]["feels_like"],
        "humidity_pct": data["main"]["humidity"],
        "weather_condition": data["weather"][0]["main"],
        "wind_speed_m_s": data["wind"]["speed"]
    }]
    
    # 2. Convert to DataFrame & Add Timestamp
    df_raw = spark.createDataFrame(weather_data).withColumn("ingested_at", current_timestamp())
    
    # 3. Save directly as a Delta Table (No File Paths Needed!)
    df_raw.write.format("delta").mode("append").saveAsTable("lahore_weather_bronze")
    
    print("Successfully written Lahore weather data to Delta Lake Bronze table!")
else:
    print(f"Failed to fetch data. HTTP Status Code: {response.status_code}")

Successfully written Lahore weather data to Delta Lake Bronze table!


In [0]:
%sql
SELECT 
    city, 
    country, 
    temp_celsius, 
    feels_like_celsius, 
    humidity_pct, 
    weather_condition, 
    wind_speed_m_s, 
    ingested_at 
FROM lahore_weather_bronze 
ORDER BY ingested_at DESC;

city,country,temp_celsius,feels_like_celsius,humidity_pct,weather_condition,wind_speed_m_s,ingested_at
Lahore,PK,33.99,39.06,52,Clear,2.06,2026-09-08T13:51:10.264Z
Lahore,PK,34.99,39.16,46,Clear,2.06,2026-09-08T13:26:42.824Z
